# 🍕 Entrenamiento Food-101 — EfficientNet-B0

**Instrucciones:**
1. Ve a `Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU (T4)`
2. Ejecuta todas las celdas en orden
3. El modelo entrenado se descarga automáticamente al final

Tiempo estimado: **~30-60 min** con T4 (vs horas en CPU)

## 1. Verificar GPU disponible

In [ ]:
import torch
print(f"CUDA disponible: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    raise RuntimeError("No se detectó GPU. Ve a Entorno de ejecución → Cambiar tipo → GPU")

## 2. Clonar repositorio

In [ ]:
!git clone https://github.com/Fernando-Alvarado-Soria/app-caloriasv2.git
%cd app-caloriasv2

## 3. Instalar dependencias

In [ ]:
!pip install torch torchvision --quiet
# Colab ya tiene torch y torchvision con CUDA, pero esto asegura la versión correcta

## 4. Configurar hiperparámetros

Puedes ajustar estos valores antes de entrenar. Con GPU puedes usar **batch_size más grande** para acelerar aún más.

In [ ]:
# Sobrescribir config para Colab (batch más grande, más workers)
import ml.config as config

# Con T4 (16GB VRAM) puedes usar batch_size 64 o incluso 128
config.BATCH_SIZE = 64
config.NUM_WORKERS = 2
config.NUM_EPOCHS = 10
config.DEVICE = "cuda"

print(f"Batch size: {config.BATCH_SIZE}")
print(f"Epochs: {config.NUM_EPOCHS}")
print(f"Device: {config.DEVICE}")

## 5. Descargar dataset Food-101

In [ ]:
from ml.download_dataset import download_food101
download_food101()

## 6. Entrenar modelo

In [ ]:
import argparse
from ml.train import train

train_args = argparse.Namespace(
    epochs=config.NUM_EPOCHS,
    batch_size=config.BATCH_SIZE,
    resume=None,
)

train(train_args)

## 7. Evaluar modelo

In [ ]:
!python -m ml.evaluate

## 8. Exportar modelo para inferencia

In [ ]:
!python -m ml.export_model

## 9. Descargar archivos entrenados

Ejecuta esta celda para descargar los archivos a tu computadora.

In [ ]:
from google.colab import files
import shutil
import os

# Crear zip con todo lo necesario
export_dir = "ml/models/export"
best_model = "ml/models/best_model.pt"

# Crear carpeta temporal para empaquetar
os.makedirs("download_pack", exist_ok=True)
if os.path.exists(export_dir):
    shutil.copytree(export_dir, "download_pack/export", dirs_exist_ok=True)
if os.path.exists(best_model):
    shutil.copy2(best_model, "download_pack/best_model.pt")

shutil.make_archive("modelo_entrenado", "zip", "download_pack")
files.download("modelo_entrenado.zip")
print("✅ Descarga iniciada. Copia los archivos a ml/models/ en tu proyecto local.")

## 10. (Opcional) Guardar en Google Drive

Si prefieres guardar directamente en Drive en vez de descargar.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import shutil
dest = '/content/drive/MyDrive/app-caloriasv2-models/'
os.makedirs(dest, exist_ok=True)

# Copiar modelo exportado
if os.path.exists('ml/models/export'):
    shutil.copytree('ml/models/export', f'{dest}/export', dirs_exist_ok=True)
if os.path.exists('ml/models/best_model.pt'):
    shutil.copy2('ml/models/best_model.pt', dest)

print(f'✅ Modelo guardado en Google Drive: {dest}')